# Tutorial 03

*   Training neural networks in PyTorch
*   Debugging and analyzing overfitting/underfitting

## Neural Networks 

*   Input (features): **x**
*   Output (labels/target values): **y**
*   We want to find a function **f** that will approximate **f(x) = y** and can be run on new data when **y** is unknown

Neural networks approximate the function **f** by composition of functions like

$$ f(x) = f^{(N)}(...f^{(2)}(f^{(1)}(x)))$$

where N is number of layers of the neural network

## Multilayer Perceptron (MLP)

Let's define an ${ f^{(i)}(x) }$ function like

$$ f^{(i)}(x) = \phi(\textbf{W}x + b) $$

where ${x \in \mathbb{R}^{D} }$ is an input vector, ${ \textbf{W} \in \mathbb{R}^{H \times D} }$ is weight matrix, ${ b \in \mathbb{R}^{H} }$ is bias vector to be added, and ${\phi(.)}$ is an activation function.

MLPs are a class of neural network where ${ f_{(i)} }$ is defined like the above. A simple MLP with 2 layers are illustrated in Figure 1

[![MLP Figure 1](https://classic.d2l.ai/_images/mlp.svg)]()
 *Figure 1: A Simple MLP*

 Two of the most commonly used activations are *sigmoid* and *ReLU* functions

 <!-- $$ Sigmoid: \sigma(x) = \frac{1}{1 + e^{-x}} $$
 $$ ReLU: R(x) = max(0, x) $$ -->

  
[![Sigmoid and ReLU figure](https://miro.medium.com/max/1452/1*XxxiA0jJvPrHEJHD4z893g.png )]()



# Download the Dataset
We are using cats vs. dogs dataset used in Tutorial 01. You may download it from [here](https://drive.usercontent.google.com/download?id=1sKjxoH-MQl3CBVEm6C0EHwV3rqtXOqWq&export=download&authuser=0) and extract it to your working folder. Alternatively, you can just run the cell below to download and extract it. If you are not running the notebook on Google Colab, make sure that you have gdown by using the command `pip install gdown`. If you are on Windows, the cell won't work. You need to download and extract the data from the link. 

In [ ]:
import torch
from tqdm.notebook import tqdm
import urllib.request
import zipfile
import torch_directml

In [12]:
!gdown --id 1sKjxoH-MQl3CBVEm6C0EHwV3rqtXOqWq
!tar -xf swaroopgrs-datasets-dogscats-1.tar

C:\Users\anabo\Documents\Master\SP26\DL\git\.venv\Lib\site-packages\gdown\__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1sKjxoH-MQl3CBVEm6C0EHwV3rqtXOqWq
From (redirected): https://drive.google.com/uc?id=1sKjxoH-MQl3CBVEm6C0EHwV3rqtXOqWq&confirm=t&uuid=d1cfb3fb-39f5-41b6-9a43-98cfc4461cad
To: c:\Users\anabo\Documents\Master\SP26\DL\git\series3\swaroopgrs-datasets-dogscats-1.tar

  0%|          | 0.00/887M [00:00<?, ?B/s]
  0%|          | 524k/887M [00:00<03:02, 4.87MB/s]
  0%|          | 1.05M/887M [00:00<03:23, 4.36MB/s]
  0%|          | 1.57M/887M [00:00<03:56, 3.74MB/s]
  0%|          | 2.10M/887M [00:00<04:22, 3.37MB/s]
  0%|          | 2.62M/887M [00:00<04:15, 3.46MB/s]
  0%|          | 3.67M/887M [00:00<03:33, 4.13MB/s]
  0%|          | 4.19M/887M [00:01<03:42, 3.97MB/s]
  1%|          | 5.2

# Training neural networks
We have a set of input vectors $x$ (features) and desired targets (value / class label) $y$  
We want to find a function $f$ such that $f(x)=y$

Basic training algorithm:  
**Input:**
- features $x$, targets $y$  
- neural network model with randomly initialized parameters $\theta$: $f(x; \theta)$  
- learning_rate $\alpha$  
- loss function $L$  

**Output:**
- optimized parameters $\theta$

**Algorithm**  
>for epoch in 1..n_epochs do
>>     for (minibatch_x, minibatch_y) in dataset do
>>>        compute predicted y' = f(x; theta)
>>>        compute loss = L(y', minibatch_y)
>>>        compute gradients = gradient(L, theta)
>>>        update parameters theta <- theta - learning_rate * gradients

First, set device variable to `cuda` if there is an availabe GPU, otherwise it will be set to `cpu`.

In [23]:

try:
    import torch_directml
    device = torch_directml.device()
except (ImportError, RuntimeError):
    device = torch.device("cpu")

print('Device: {}'.format(device))

Device: cpu


# Dataset
We are defining the dataset object so that Pytorch shall know how to load the dataset, we are using `ImageFolder` class of Pytorch, which puts images in the subfolders of `root_dir` to different classes. Since we have two folder called "cats" and "dogs", we'll have two classes for cats and dogs.

In [14]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import Resize, ToTensor, Normalize, Compose
root_dir = 'images'

target_size = (32, 32)
transforms = Compose([Resize(target_size), # Resizes image, wraps in Composer object
                      ToTensor(),           # Converts to Tensor, scales to [0, 1] float (from [0, 255] int)
                      Normalize(mean=(0.5, 0.5, 0.5,), std=(0.5, 0.5, 0.5)), # scales to [-1.0, 1.0], for optimization purposes
                    ])

train_dataset_ = ImageFolder('train', transform=transforms) #when u pass transformers here, when u access one object it converts img to tensor
val_dataset_ = ImageFolder('valid', transform=transforms) #the data isnt loaded yet, we just know where they're located and can access them from the disk
#when the DS is too large, we cannot load it entirely to the RAM

In [15]:
train_dataset_[9]

(tensor([[[-0.0275, -0.0431, -0.1216,  ...,  0.1922,  0.2784,  0.2078],
          [-0.0431, -0.0588, -0.1216,  ...,  0.2157,  0.3098,  0.2471],
          [-0.0431, -0.0745, -0.1294,  ...,  0.1922,  0.3098,  0.2549],
          ...,
          [ 0.6549,  0.6314,  0.6549,  ...,  0.6392,  0.4588,  0.0196],
          [ 0.6627,  0.5137,  0.5608,  ...,  0.7020,  0.6157,  0.2471],
          [ 0.6706,  0.5451,  0.4824,  ...,  0.7333,  0.6549,  0.5294]],
 
         [[-0.5608, -0.5765, -0.5922,  ..., -0.3725, -0.2706, -0.3255],
          [-0.5765, -0.5843, -0.5922,  ..., -0.3490, -0.2392, -0.2863],
          [-0.5765, -0.5843, -0.5608,  ..., -0.3725, -0.2471, -0.3020],
          ...,
          [ 0.2235,  0.1765,  0.1765,  ...,  0.2471,  0.0196, -0.4980],
          [ 0.2078, -0.0039,  0.0353,  ...,  0.3098,  0.2078, -0.2471],
          [ 0.1608, -0.0118, -0.0510,  ...,  0.3412,  0.2863,  0.1059]],
 
         [[-0.5686, -0.5765, -0.6000,  ..., -0.3647, -0.2706, -0.3333],
          [-0.5843, -0.5843,

In [16]:
print('Number of training samples: {}'.format(len(train_dataset_)))
print('Number of validation samples: {}'.format(len(val_dataset_)))

Number of training samples: 23000
Number of validation samples: 2000


# Load the Dataset
Instead of loading opening image files one by one, we'll load them to memory in the beginning so that the training would be faster.

In [17]:
class RAMDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset):
        data = []
        for sample in tqdm(dataset):
            data.append(sample)
        self.n = len(dataset)
        self.data = data
        
    def __getitem__(self, ind):
        return self.data[ind]

    def set_transform(self, transform):
        self.transform = transform #torch vision transforms have a wide range; we need to use them when we train the data we need it in transform for to apply math to it
    
    def __len__(self):
        return self.n

In [25]:
train_dataset = RAMDatasetWrapper(train_dataset_)

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

# Random Sample
Let's pick a random sample from dataset and visualize it with matplotlib

In [26]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

index = np.random.randint(0, high=len(train_dataset))
img_to_show = train_dataset[index][0].mul(0.5).add(0.5).permute(1,2,0).cpu().numpy()
label_sample = train_dataset[index][1]
label_dict = {0: 'cat', 1: 'dog'}

plt.imshow(img_to_show)
plt.title('It is a {}'.format(label_dict[label_sample]))

NameError: name 'train_dataset' is not defined

# Dataloaders
Defining the dataloaders for training and validation datasets. This object will do loading of the dataset with desired batch size and shuffles the data.

In [27]:
from torch.utils.data import DataLoader
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2) #num_workers = n - how many threads in background for efficient loading

NameError: name 'train_dataset' is not defined

In [ ]:
# Same for validation dataset
val_dataset = RAMDatasetWrapper(val_dataset_)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Define the Models
We are declearing the one linear model and one MLP model that we are going to use through this tutorial by inhereting the Pytorch's `nn.Module` class. The input images are converted into 1D vectors in both models, then a binary prediction is made to conclude that the image is cat or dog 

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module): # u need to have init and forward: in init u tell which layers u need with what characteristics (conv, linear,etc.), in forward u tell the model in what order it needs to apply the layers to the output
    def __init__(self, input_dim):
        super(LinearModel, self).__init__()
        self.fc = nn.Linear(input_dim, 2, bias=True) # outputs 2 values - score for cat and score for dog
        
    def forward(self, input):
        out = input.view(input.size(0), -1) # convert batch_size x 3 x imH x imW to batch_size x (3*imH*imW) #make sure we have a flatten layer here
        out = self.fc(out) # Applies out = input * A + b. A, b are parameters of nn.Linear that we want to learn
        return out
    
    
    
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(MLPModel, self).__init__()
        
        #innefficient way of doing if we have 50+ layers
        '''
        self.fc1= nn.Linear(input_dim, hidden_dim) #1st input layer
        self.act1 =nn.ReLU() #activation func, here ReLU
        
        self.fc2= nn.Linear(hidden_dim, hidden_dim) #make sure that the input size is compatible with the 1st layer; output layer has the size of the hiddden_dim
        self.act2 =nn.ReLU() 
        
        self.fc3= nn.Linear(hidden_dim, hidden_dim) 
        self.act3 =nn.ReLU() 
        
        self.fc1= nn.Linear(hidden_dim, hidden_dim) #last layer's output is the output of the nn: for our problem of a classification task, it is of size 2
        #now we just initialized it
        '''
        layers= []
        
        for i in range(num_layers) #num_layers as input, place it in the init params
        
            if i == 0: #1st layer
                layers.append(nn.Linear(input_dim,hidden_dim))
                layers.append(nn.ReLU())
                
            elif i == num_layers-1: #last layer
                layers.append(nn.Linear(hidden_dim,2))
                
            else:#intermediate layers
                layers.append(nn.Linear(hidden_dim,hidden_dim))
                layers.append(nn.ReLU())
        
        self.net = nn.Sequential(*layers) #put layers inside in order
        
        
    def forward(self, input):
        #innefficient way, same
        '''
        out= input.view(input.size(0),-1) #convert batch_size x 3 x imH x imW to...
        out=self.fc1(out)
        out= self.act1(out)
        
        out=self.fc2(out)
        out= self.act2(out)
        
        out=self.fc3(out)
        out= self.act3(out)
        
        out=self.fc2(out) #why fc2 out? mistake?
        '''
        out= input.view(input.size(0),-1) #convert batch_size x 3 x imH x imW to...
        out=self.net(out)
        return out

In [ ]:
#to make sure everything is correct, we can init a model
model = MLPModel(10,30,5)
model

# Training Functions
We are declaring the functions that we'll use for training. `train_epoch` function performs the the training for one epoch, `train` function trains the model for the desired number of epochs. 

In [ ]:
import numpy as np

def train_epoch(model, train_dataloader, optimizer, loss_fn):
    losses = []
    correct_predictions = 0
    # Iterate mini batches over training dataset
    for images, labels in tqdm(train_dataloader):
        # move tensors to current device
     
         
        # Run predictions
        #output= ...
        
        # Set gradients to zero
        # https://pytorch.org/docs/stable/optim.html
        # https://discuss.pytorch.org/t/why-do-we-need-to-set-the-gradients-manually-to-zero-in-pytorch/4903/20
        
        
        # Compute loss
        #loss= ...
        
        # Backpropagate (compute gradients)
        
        
        # Make an optimization step (update parameters)
        
        
        # Log metrics
        losses.append(loss.item())
        predicted_labels = output.argmax(dim=1)
        correct_predictions += (predicted_labels == labels).sum().item()
    accuracy = 100.0 * correct_predictions / len(train_dataloader.dataset)
    # Return loss values for each iteration and accuracy
    mean_loss = np.array(losses).mean()
    return mean_loss, accuracy

In [ ]:
def evaluate(model, dataloader, loss_fn):
    losses = []
    correct_predictions = 0
    with torch.no_grad():
        for images, labels in dataloader:
            
            #move tensors to current device
            
            
            # Run predictions
            #output = ...
            
            
            # Compute loss
            #loss = ...
            
            
            predicted_labels = output.argmax(dim=1)
            correct_predictions += (predicted_labels == labels).sum().item()
            losses.append(loss.item())
    mean_loss = np.array(losses).mean()
    accuracy = 100.0 * correct_predictions / len(dataloader.dataset)
    # Return mean loss and accuracy
    return mean_loss, accuracy

In [ ]:
def train(model, train_dataloader, val_dataloader, optimizer, n_epochs, loss_function):
    # We will monitor loss functions as the training progresses
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(n_epochs):
        # set model to train mode
        
        # train the model for one epoch
        #train_loss, train_accuracy = ...
        
        # set the model to eval model
        
        # evaluate the model with validation data
        #val_loss, val_accuracy = ...
        
        # append losses and accuracies
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print('Epoch {}/{}: train_loss: {:.4f}, train_accuracy: {:.4f}, val_loss: {:.4f}, val_accuracy: {:.4f}'.format(epoch+1, n_epochs,
                                                                                                      train_losses[-1],
                                                                                                      train_accuracies[-1],
                                                                                                      val_losses[-1],
                                                                                                      val_accuracies[-1]))
    return train_losses, val_losses, train_accuracies, val_accuracies

# Plotting
We are defining the plot function to plot the loss vs. epoch and accuracy vs. epoch graphs. The loss and accuracy for training and validation dataset are plotted on the same graph. 

In [ ]:
def plot(n_epochs, train_losses, val_losses, train_accuracies, val_accuracies):
    plt.figure()
    plt.plot(np.arange(n_epochs), train_losses)
    plt.plot(np.arange(n_epochs), val_losses)
    plt.legend(['train_loss', 'val_loss'])
    plt.xlabel('epoch')
    plt.ylabel('loss value')
    plt.title('Train/val loss')

    plt.figure()
    plt.plot(np.arange(n_epochs), train_accuracies)
    plt.plot(np.arange(n_epochs), val_accuracies)
    plt.legend(['train_acc', 'val_acc'])
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.title('Train/val accuracy')


# Linear Model training


In [ ]:
model = LinearModel(32*32*3)
model = model.to(device)

# Train the linear model

In [ ]:
# Visualize losses, accuracies

# MLP training


In [ ]:
model = MLPModel(32*32*3, 64)
model = model.to(device)

# Train the MLP model


In [ ]:
# Visualize losses, accuracies